# Laboratorio 7 — Spark MLlib
---

## 0. Configuración inicial

Importamos librerías, iniciamos la sesión de Spark y definimos rutas y parámetros globales del notebook.

In [1]:
import os
import re
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer, OneHotEncoder
from pyspark.ml.stat import Correlation
from pyspark.ml.clustering import KMeans
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import ClusteringEvaluator, RegressionEvaluator
from pyspark.ml import Pipeline

os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = (SparkSession.builder
         .appName("Lab7_SparkMLlib_ENEIC")
         .master("local[*]")
         .config("spark.driver.host", "127.0.0.1")
         .config("spark.driver.bindAddress", "127.0.0.1")
         .config("spark.sql.shuffle.partitions", "8")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
spark


In [5]:
# ------------------------------------------------------------------
# RUTAS Y PARÁMETROS GLOBALES
# ------------------------------------------------------------------
RAW_DIR = "../data/raw"          # carpeta con los .xlsx originales de la ENEIC
PARQUET_DIR = "../data/parquet"  # carpeta de salida para los datos preparados
MODEL_DIR = "../models"          # carpeta para guardar modelos ajustados
SAMPLE_MAX = 8000             # tamaño máximo de muestra SOLO para graficar
SEED = 42

os.makedirs(PARQUET_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

FILE_MAP = {
    "2025T1": {"path": f"{RAW_DIR}/Personas_2025_T1.xlsx", "anio": 2025, "trim_cal": 1, "uso": "train"},
    "2025T2": {"path": f"{RAW_DIR}/Personas_2025_T2.xlsx", "anio": 2025, "trim_cal": 2, "uso": "train"},
    "2025T3": {"path": f"{RAW_DIR}/Personas_2025_T3.xlsx", "anio": 2025, "trim_cal": 3, "uso": "train"},
    "2025T4": {"path": f"{RAW_DIR}/Personas_2025_T4.xlsx", "anio": 2025, "trim_cal": 4, "uso": "train"},
    "2026T1": {"path": f"{RAW_DIR}/Personas_2026_T1.xlsx", "anio": 2026, "trim_cal": 1, "uso": "test"},
}

# Columnas originales que se conservan del archivo fuente (ver tabla "Variables que utilizará")
REQUIRED_COLS = [
    "NUM_HOGAR", "NUM_PERSONA",      # auditoría de registros
    "FACTOR",                         # diseño muestral (no se usa para ponderar en este lab)
    "ANIO", "TRIMESTRE",              # auditoría de la fuente (TRIMESTRE NO es el calendario)
    "OCUPADOS",                       # filtro
    "P05C16",                         # categoria_ocupacional (filtro + predictor)
    "P05D01",                         # salario_mensual (variable objetivo)
    "P02A03",                         # edad
    "P05C07A", "P05C07B",             # antiguedad_anios, antiguedad_meses
    "P05H01A",                        # horas_semanales
    "P03A03A",                        # nivel_educativo
    "DOMINIO",                        # dominio
]


## 1. Carga, armonización y calidad de datos — (5 pts)

**Enfoque:**
1. Cada archivo Excel se lee con `pandas`/`openpyxl` **como texto (`dtype=str`)**. Esto es
   clave: en la ENEIC un mismo código puede llegar como número (`1`) o como texto con
   decimales espurios (`"1.0"`), y leer todo como texto nos permite normalizarlo de forma
   uniforme *antes* de convertir a Spark, en vez de arrastrar inconsistencias de tipo.
2. Se seleccionan únicamente las columnas requeridas (esto resuelve el problema de que el
   archivo de IV-2025 tiene 302 columnas en vez de 270: al elegir el mismo subconjunto de
   columnas en todos los archivos, `unionByName` los puede unir sin importar cuántas
   columnas tenía cada archivo originalmente).
3. Se agregan `archivo_origen`, `periodo_archivo`, `anio_archivo` y `trimestre_calendario`
   a partir del archivo de procedencia (NO a partir de la columna `TRIMESTRE`, que trae
   valores inconsistentes entre archivos: 2, 3, 4, 5 y 6, y que en II-2025 mezcla 3 y 2).
4. Se homologan los códigos categóricos a una representación de texto consistente
   (quitando sufijos `".0"` que aparecen cuando el valor llegó como numérico).
5. Se hace `unionByName` de los cuatro trimestres de 2025.


In [ ]:
def leer_archivo_personas(periodo, path, anio, trim_cal):
    """Lee un archivo de Personas de la ENEIC, selecciona columnas requeridas y
    agrega las columnas de identificación de período. Devuelve (spark_df, n_original)."""
    pdf = pd.read_excel(path, dtype=str)
    faltan = [c for c in REQUIRED_COLS if c not in pdf.columns]
    if faltan:
        raise ValueError(f"[{periodo}] Faltan columnas requeridas: {faltan}")

    n_original = len(pdf)
    pdf = pdf[REQUIRED_COLS].copy()

    sdf = spark.createDataFrame(pdf)
    sdf = (sdf
           .withColumn("archivo_origen", F.lit(periodo))
           .withColumn("periodo_archivo", F.lit(periodo))
           .withColumn("anio_archivo", F.lit(int(anio)))
           .withColumn("trimestre_calendario", F.lit(int(trim_cal))))
    return sdf, n_original


def normalizar_codigo(colname):
    """Convierte una columna de texto a una representación de código consistente:
    quita espacios y el sufijo '.0' que aparece cuando el dato llegó como numérico."""
    c = F.trim(F.col(colname).cast("string"))
    c = F.regexp_replace(c, r"\.0$", "")
    return F.when((c == "") | c.isNull(), None).otherwise(c)


# --- Cargar los 4 trimestres de 2025 ---
dfs_2025 = []
registros_originales = {}
for periodo in ["2025T1", "2025T2", "2025T3", "2025T4"]:
    info = FILE_MAP[periodo]
    sdf, n_orig = leer_archivo_personas(periodo, info["path"], info["anio"], info["trim_cal"])
    dfs_2025.append(sdf)
    registros_originales[periodo] = n_orig

df_2025 = dfs_2025[0]
for d in dfs_2025[1:]:
    df_2025 = df_2025.unionByName(d)

df_2025.persist()
print("Registros originales por archivo (2025):", registros_originales)
print("Total tras unionByName:", df_2025.count())


In [ ]:
# --- Esquema y 5 registros de las columnas seleccionadas ---
df_2025.printSchema()
df_2025.select(*REQUIRED_COLS, "archivo_origen", "periodo_archivo",
               "anio_archivo", "trimestre_calendario").show(5, truncate=False)


In [ ]:
# --- Cargar también I-2026 (se preparará igual, pero se guarda por separado: es TEST) ---
info_2026 = FILE_MAP["2026T1"]
df_2026, n_orig_2026 = leer_archivo_personas("2026T1", info_2026["path"],
                                              info_2026["anio"], info_2026["trim_cal"])
df_2026.persist()
registros_originales["2026T1"] = n_orig_2026
print("Registros originales I-2026:", n_orig_2026)


### Homologación de tipos y armonización de códigos

Se castean las variables numéricas a `double` y se normalizan las variables categóricas
(`P05C16`, `P03A03A`, `DOMINIO`, `OCUPADOS`, `NUM_HOGAR`, `NUM_PERSONA`) a texto consistente
con `normalizar_codigo`. El código educativo `"0"` (ninguno) se conserva explícitamente y
**no** se trata como faltante.


In [ ]:
def armonizar_tipos(sdf):
    sdf = (sdf
        .withColumn("NUM_HOGAR", normalizar_codigo("NUM_HOGAR"))
        .withColumn("NUM_PERSONA", normalizar_codigo("NUM_PERSONA"))
        .withColumn("OCUPADOS", normalizar_codigo("OCUPADOS"))
        .withColumn("ocupado", F.col("OCUPADOS"))
        .withColumn("categoria_ocupacional", normalizar_codigo("P05C16"))
        .withColumn("nivel_educativo", normalizar_codigo("P03A03A"))
        .withColumn("dominio", normalizar_codigo("DOMINIO"))
        .withColumn("salario_mensual", F.col("P05D01").cast(T.DoubleType()))
        .withColumn("edad", F.col("P02A03").cast(T.DoubleType()))
        .withColumn("antiguedad_anios", F.col("P05C07A").cast(T.DoubleType()))
        .withColumn("antiguedad_meses", F.col("P05C07B").cast(T.DoubleType()))
        .withColumn("horas_semanales", F.col("P05H01A").cast(T.DoubleType()))
        .withColumn("FACTOR", F.col("FACTOR").cast(T.DoubleType()))
        .withColumn("ANIO", F.col("ANIO").cast(T.IntegerType()))
        .withColumn("TRIMESTRE", F.col("TRIMESTRE").cast(T.IntegerType()))
    )
    return sdf

df_2025_tip = armonizar_tipos(df_2025)
df_2026_tip = armonizar_tipos(df_2026)
df_2025_tip.persist()
df_2026_tip.persist()
df_2025_tip.select("categoria_ocupacional", "nivel_educativo", "dominio",
                    "salario_mensual", "edad", "antiguedad_anios",
                    "antiguedad_meses", "horas_semanales").show(5)


### Faltantes por variable (antes de aplicar filtros)

Se calcula cantidad y porcentaje de nulos/vacíos para cada variable clave, **sobre la unión
de 2025 sin filtrar todavía**.


In [ ]:
KEY_COLS = ["salario_mensual", "edad", "antiguedad_anios", "antiguedad_meses",
            "horas_semanales", "nivel_educativo", "categoria_ocupacional",
            "dominio", "ocupado"]

n_total_2025 = df_2025_tip.count()
filas_faltantes = []
for c in KEY_COLS:
    n_null = df_2025_tip.filter(F.col(c).isNull() | (F.col(c).cast("string") == "")).count()
    filas_faltantes.append((c, n_null, round(100 * n_null / n_total_2025, 2)))

faltantes_pdf = pd.DataFrame(filas_faltantes, columns=["variable", "n_faltantes", "pct_faltantes"])
faltantes_pdf


### Filtros de población analítica

Se aplican en un **orden fijo** para poder contabilizar cuántos registros se excluyen en
cada paso. Nótese el uso explícito de `isnan()` (además de `isNotNull()`): en Spark, un
valor `NaN` se considera *mayor* que cualquier otro número, por lo que un filtro como
`salario_mensual > 0` **no** excluye los `NaN` por sí solo. Por eso se define un helper
`es_finito` que exige que el valor no sea nulo, no sea `NaN` y no sea `+/-Infinito`.


In [ ]:
def es_finito(colname):
    c = F.col(colname)
    return c.isNotNull() & (~F.isnan(c)) & (c != float("inf")) & (c != float("-inf"))


def preparar_poblacion_analitica(sdf, etiqueta):
    """Aplica los filtros de la población analítica en orden fijo, registrando
    cuántos registros se excluyen en cada paso."""
    pasos = [("0_inicial", sdf.count())]
    d = sdf

    d = d.filter(es_finito("edad") & (F.col("edad") >= 15))
    pasos.append(("1_edad>=15_finita", d.count()))

    d = d.filter(F.col("ocupado") == "1")
    pasos.append(("2_ocupado==1", d.count()))

    d = d.filter(F.col("categoria_ocupacional").isin(["1", "2", "3", "4"]))
    pasos.append(("3_asalariado_P05C16_1a4", d.count()))

    d = d.filter(es_finito("salario_mensual") & (F.col("salario_mensual") > 0))
    pasos.append(("4_salario_finito_positivo", d.count()))

    d = d.filter(es_finito("antiguedad_meses")
                 & (F.col("antiguedad_meses") == F.floor(F.col("antiguedad_meses")))
                 & (F.col("antiguedad_meses") >= 0) & (F.col("antiguedad_meses") <= 11))
    pasos.append(("5_meses_0_a_11_enteros", d.count()))

    d = d.filter(es_finito("antiguedad_anios") & (F.col("antiguedad_anios") >= 0))
    pasos.append(("6_antiguedad_anios>=0", d.count()))

    d = d.withColumn("antiguedad", F.col("antiguedad_anios") + F.col("antiguedad_meses") / F.lit(12.0))
    d = d.filter(F.col("antiguedad") <= F.col("edad"))
    pasos.append(("7_antiguedad<=edad", d.count()))

    d = d.filter(es_finito("horas_semanales")
                 & (F.col("horas_semanales") > 0) & (F.col("horas_semanales") <= 168))
    pasos.append(("8_horas_0_a_168", d.count()))

    # Categóricas ausentes o no reconocidas -> "DESCONOCIDO" (0 en nivel_educativo NO es faltante)
    # Fuente: Diccionario Personas ENEIC (I-2025, II-2025, III-2025, IV-2025, I-2026) — códigos idénticos en los 5 archivos
    codigos_educativos_validos = ["0", "1", "2", "3", "4", "5", "6", "7"]
    # 0=NINGUNO, 1=PREPRIMARIA, 2=PRIMARIA, 3=BÁSICO, 4=DIVERSIFICADO, 5=SUPERIOR, 6=MAESTRÍA, 7=DOCTORADO
    dominios_validos = ["1", "2", "3"]
    # 1=Urbano Metropolitano, 2=Resto Urbano, 3=Rural Nacional

    d = d.withColumn("nivel_educativo",
                      F.when(F.col("nivel_educativo").isin(codigos_educativos_validos),
                             F.col("nivel_educativo"))
                       .otherwise(F.lit("DESCONOCIDO")))
    d = d.withColumn("dominio",
                      F.when(F.col("dominio").isin(dominios_validos), F.col("dominio"))
                       .otherwise(F.lit("DESCONOCIDO")))

    print(f"--- Pasos de filtrado ({etiqueta}) ---")
    for nombre, n in pasos:
        print(f"  {nombre}: {n}")

    return d, pasos


df_2025_analitico, pasos_2025 = preparar_poblacion_analitica(df_2025_tip, "2025 - train/dev")
df_2026_analitico, pasos_2026 = preparar_poblacion_analitica(df_2026_tip, "2026 - test final")

df_2025_analitico.persist()
df_2026_analitico.persist()

pasos_2025_pdf = pd.DataFrame(pasos_2025, columns=["paso", "n_registros"])
pasos_2025_pdf["n_excluidos"] = pasos_2025_pdf["n_registros"].shift(1) - pasos_2025_pdf["n_registros"]
pasos_2025_pdf


In [ ]:
# Registros por archivo antes y después de los filtros analíticos.
antes_2025 = pd.DataFrame(
    sorted(registros_originales.items()), columns=["periodo_archivo", "registros_originales"]
)
df_analitico_todos = df_2025_analitico.unionByName(df_2026_analitico)
despues_todos = (df_analitico_todos.groupBy("periodo_archivo")
                .count()
                .withColumnRenamed("count", "registros_post_filtro")
                .orderBy("periodo_archivo")
                .toPandas())
conteo_archivos = antes_2025.merge(despues_todos, on="periodo_archivo", how="left")
conteo_archivos["registros_excluidos"] = (
    conteo_archivos["registros_originales"] - conteo_archivos["registros_post_filtro"]
)
conteo_archivos


**Nota:** las variables categóricas se validan contra los códigos definidos en el
diccionario de Personas de la ENEIC usado para estos archivos: `nivel_educativo` acepta
los códigos `0` a `7` (donde `0` significa ninguno y no es faltante) y `dominio` acepta
`1` a `3`. Los valores ausentes o no reconocidos se marcan como `DESCONOCIDO`, sin
convertirlos arbitrariamente a cero.

### Verificación de unicidad de `(periodo_archivo, NUM_HOGAR, NUM_PERSONA)`

In [ ]:
claves = df_2025_analitico.groupBy("periodo_archivo", "NUM_HOGAR", "NUM_PERSONA").count()
duplicados = claves.filter(F.col("count") > 1)
n_dup = duplicados.count()
print(f"Combinaciones de clave con más de 1 registro: {n_dup}")

if n_dup > 0:
    # Inspeccionar si son repeticiones EXACTAS (todas las columnas iguales) o registros en conflicto
    llaves_dup = duplicados.select("periodo_archivo", "NUM_HOGAR", "NUM_PERSONA")
    filas_dup = df_2025_analitico.join(llaves_dup, on=["periodo_archivo", "NUM_HOGAR", "NUM_PERSONA"])
    n_filas_dup = filas_dup.count()
    n_filas_dup_exactas = filas_dup.distinct().count()
    print(f"Filas involucradas: {n_filas_dup} | Filas distintas tras comparar TODAS las columnas: {n_filas_dup_exactas}")
    if n_filas_dup_exactas < n_filas_dup:
        print("=> Hay repeticiones EXACTAS de fila completa (posible doble carga).")
    if n_filas_dup_exactas == n_filas_dup:
        print("=> Son registros EN CONFLICTO (misma clave, columnas distintas): requiere investigación manual.")
    filas_dup.orderBy("periodo_archivo", "NUM_HOGAR", "NUM_PERSONA").show(20, truncate=False)
else:
    print("No se encontraron combinaciones duplicadas de (periodo_archivo, NUM_HOGAR, NUM_PERSONA).")


### Guardar en Parquet

In [ ]:
df_2025_analitico.write.mode("overwrite").parquet(f"{PARQUET_DIR}/personas_2025_analitico.parquet")
df_2026_analitico.write.mode("overwrite").parquet(f"{PARQUET_DIR}/personas_2026_analitico.parquet")
print("Guardado en:", PARQUET_DIR)


### Respuestas — Ejercicio 1

**¿Por qué IV de 2025 no puede apilarse por posición de columnas con los otros archivos?**
Porque tiene 302 columnas originales frente a las 270 de los demás trimestres: el orden y
significado de las columnas por posición no coincide entre archivos. Apilar por posición
(`union`) mezclaría columnas distintas bajo el mismo nombre de columna. Por eso se usa
`unionByName` sobre un subconjunto fijo de columnas seleccionadas por **nombre**, no por
posición.

**¿Qué diferencia existe entre un dato ausente porque la pregunta no corresponde y una
respuesta no registrada?**
Un dato ausente "por no corresponder" ocurre cuando la pregunta no aplica a esa persona
dado el flujo del cuestionario (por ejemplo, preguntas de antigüedad laboral no se le hacen
a alguien que no trabaja); es un vacío *estructural* y esperado. Una respuesta no registrada
es un vacío en una pregunta que sí correspondía a la persona, pero que no quedó capturada
(el encuestado no respondió, error de captura, etc.); es un faltante *real* que sí afecta la
calidad del dato para ese registro. Tratarlos igual (por ejemplo, imputando ambos a 0)
distorsiona el análisis.

**¿Por qué una persona observada en dos períodos no debe eliminarse como duplicado del
conjunto longitudinal?**
Porque la ENEIC tiene diseño de panel rotativo: la misma persona puede ser entrevistada en
trimestres consecutivos como parte del diseño muestral. Esas observaciones repetidas de la
misma persona en distintos períodos son observaciones legítimas (representan su situación en
cada corte), no errores de captura. Un verdadero duplicado sería la misma persona repetida
dentro de un mismo `periodo_archivo`.

**¿Por qué el número de registros de la base filtrada no representa a todos los trabajadores
del país?**
Porque (a) es una muestra encuestal, no un censo, y solo mediante el factor de expansión
(`FACTOR`) se puede proyectar a la población total; en este laboratorio se trabaja de forma
**no ponderada**, así que los conteos describen únicamente a las personas encuestadas que
cumplen los filtros, y (b) los filtros aplicados (edad, ocupación, asalariados, salario
positivo, consistencia de antigüedad/horas) excluyen deliberadamente a otros grupos de la
población económicamente activa (independientes, no ocupados, registros inconsistentes).


## 2. Estadística descriptiva y preguntas de exploración — (5 pts)

Todas las métricas y estadísticas se calculan sobre el **conjunto completo** de la
población analítica de 2025 (`df_2025_analitico`). Para graficar se toma una muestra de
hasta `SAMPLE_MAX` registros cuando se necesitan datos a nivel de fila; los gráficos de
conteos/medianas por grupo se construyen sobre **agregados** (no requieren muestreo).


In [ ]:
NUM_COLS_DESC = ["salario_mensual", "edad", "antiguedad", "horas_semanales"]

def resumen_descriptivo(sdf, cols):
    filas = []
    n_total = sdf.count()
    for c in cols:
        agg = sdf.select(
            F.count(c).alias("n"),
            F.mean(c).alias("media"),
            F.expr(f"percentile_approx({c}, 0.5)").alias("mediana"),
            F.stddev(c).alias("sd"),
            F.min(c).alias("min"),
            F.max(c).alias("max"),
            F.expr(f"percentile_approx({c}, 0.25)").alias("p25"),
            F.expr(f"percentile_approx({c}, 0.75)").alias("p75"),
            F.expr(f"percentile_approx({c}, 0.95)").alias("p95"),
        ).first()
        filas.append((c,) + tuple(agg))
    return pd.DataFrame(filas, columns=["variable", "n", "media", "mediana", "sd",
                                         "min", "max", "p25", "p75", "p95"])

resumen_2025 = resumen_descriptivo(df_2025_analitico, NUM_COLS_DESC)
resumen_2025


In [ ]:
# --- Muestra para graficar (hasta SAMPLE_MAX registros; las metricas ya se calcularon arriba) ---
n_analitico_2025 = df_2025_analitico.count()
frac = min(1.0, SAMPLE_MAX / n_analitico_2025)
muestra_2025_pdf = df_2025_analitico.sample(withReplacement=False, fraction=frac, seed=SEED).toPandas()
print(f"Muestra para graficar: {len(muestra_2025_pdf)} de {n_analitico_2025} registros analíticos")


### ¿Cómo se distribuyen los registros entre categorías ocupacionales, niveles educativos y dominios?

In [ ]:
# Distribuciones: se calculan como AGREGADOS sobre el conjunto completo (no la muestra)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

conteo_cat = (df_2025_analitico.groupBy("categoria_ocupacional").count()
              .orderBy("categoria_ocupacional").toPandas())
axes[0].bar(conteo_cat["categoria_ocupacional"], conteo_cat["count"])
axes[0].set_title("Registros por categoría ocupacional")
axes[0].set_xlabel("categoria_ocupacional")

conteo_edu = (df_2025_analitico.groupBy("nivel_educativo").count()
              .orderBy("nivel_educativo").toPandas())
axes[1].bar(conteo_edu["nivel_educativo"], conteo_edu["count"])
axes[1].set_title("Registros por nivel educativo")
axes[1].set_xlabel("nivel_educativo")
axes[1].tick_params(axis="x", rotation=45)

conteo_dom = (df_2025_analitico.groupBy("dominio").count()
              .orderBy("dominio").toPandas())
axes[2].bar(conteo_dom["dominio"], conteo_dom["count"])
axes[2].set_title("Registros por dominio")
axes[2].set_xlabel("dominio")

plt.tight_layout()
plt.show()


In [ ]:
top_cat = conteo_cat.sort_values("count", ascending=False).iloc[0]
top_edu = conteo_edu.sort_values("count", ascending=False).iloc[0]
top_dom = conteo_dom.sort_values("count", ascending=False).iloc[0]
print("Interpretación:")
print(f"- La categoría ocupacional con más registros es {top_cat['categoria_ocupacional']} ({int(top_cat['count']):,}).")
print(f"- El nivel educativo con más registros es {top_edu['nivel_educativo']} ({int(top_edu['count']):,}).")
print(f"- El dominio con más registros es {top_dom['dominio']} ({int(top_dom['count']):,}).")
print("- Estas distribuciones ayudan a identificar grupos con poca muestra antes de interpretar comparaciones por categoría.")


### ¿El salario presenta una distribución simétrica o asimétrica? ¿Qué diferencia existe entre su media y su mediana?

In [ ]:
media_sal = resumen_2025.loc[resumen_2025["variable"] == "salario_mensual", "media"].values[0]
mediana_sal = resumen_2025.loc[resumen_2025["variable"] == "salario_mensual", "mediana"].values[0]
print(f"Media: {media_sal:,.2f} | Mediana: {mediana_sal:,.2f} | Diferencia: {media_sal - mediana_sal:,.2f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(muestra_2025_pdf["salario_mensual"], bins=50)
axes[0].axvline(media_sal, color="red", linestyle="--", label="media")
axes[0].axvline(mediana_sal, color="green", linestyle="--", label="mediana")
axes[0].set_title("Salario mensual (escala original, quetzales)")
axes[0].legend()

axes[1].hist(np.log(muestra_2025_pdf["salario_mensual"]), bins=50)
axes[1].set_title("log(Salario mensual) — SOLO para visualizar\n(el modelado usa quetzales, no log)")
plt.tight_layout()
plt.show()


In [ ]:
sesgo = "a la derecha" if media_sal > mediana_sal else "a la izquierda o aproximadamente simétrica"
print("Interpretación:")
print(f"- La media salarial es Q{media_sal:,.2f} y la mediana es Q{mediana_sal:,.2f}.")
print(f"- La diferencia media - mediana es Q{media_sal - mediana_sal:,.2f}, lo que sugiere una distribución {sesgo}.")
print("- Si la media supera claramente a la mediana, pocos salarios altos elevan el promedio; por eso la mediana es útil para comparaciones entre grupos.")


### ¿Cómo varía el salario mediano entre niveles educativos y categorías ocupacionales?

In [ ]:
mediana_por_edu = (df_2025_analitico.groupBy("nivel_educativo")
                    .agg(F.expr("percentile_approx(salario_mensual, 0.5)").alias("salario_mediano"),
                         F.count("*").alias("n"))
                    .orderBy("nivel_educativo").toPandas())

mediana_por_cat = (df_2025_analitico.groupBy("categoria_ocupacional")
                    .agg(F.expr("percentile_approx(salario_mensual, 0.5)").alias("salario_mediano"),
                         F.count("*").alias("n"))
                    .orderBy("categoria_ocupacional").toPandas())

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].bar(mediana_por_edu["nivel_educativo"], mediana_por_edu["salario_mediano"])
axes[0].set_title("Salario mediano por nivel educativo")
axes[0].tick_params(axis="x", rotation=45)

axes[1].bar(mediana_por_cat["categoria_ocupacional"], mediana_por_cat["salario_mediano"])
axes[1].set_title("Salario mediano por categoría ocupacional")
plt.tight_layout()
plt.show()

display(mediana_por_edu)
display(mediana_por_cat)


In [ ]:
edu_max = mediana_por_edu.sort_values("salario_mediano", ascending=False).iloc[0]
edu_min = mediana_por_edu.sort_values("salario_mediano", ascending=True).iloc[0]
cat_max = mediana_por_cat.sort_values("salario_mediano", ascending=False).iloc[0]
cat_min = mediana_por_cat.sort_values("salario_mediano", ascending=True).iloc[0]
print("Interpretación:")
print(f"- El mayor salario mediano por educación aparece en nivel {edu_max['nivel_educativo']} (Q{edu_max['salario_mediano']:,.2f}; n={int(edu_max['n']):,}).")
print(f"- El menor salario mediano por educación aparece en nivel {edu_min['nivel_educativo']} (Q{edu_min['salario_mediano']:,.2f}; n={int(edu_min['n']):,}).")
print(f"- Por categoría ocupacional, la mediana más alta es {cat_max['categoria_ocupacional']} (Q{cat_max['salario_mediano']:,.2f}; n={int(cat_max['n']):,}) y la más baja es {cat_min['categoria_ocupacional']} (Q{cat_min['salario_mediano']:,.2f}; n={int(cat_min['n']):,}).")
print("- Las diferencias deben leerse junto con n: grupos pequeños producen medianas menos estables.")


### ¿Cómo cambian el tamaño de la muestra analítica y el salario mediano entre trimestres?

In [ ]:
por_trimestre = (df_2025_analitico.groupBy("periodo_archivo")
                  .agg(F.count("*").alias("n_registros"),
                       F.expr("percentile_approx(salario_mensual, 0.5)").alias("salario_mediano"))
                  .orderBy("periodo_archivo").toPandas())

fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.bar(por_trimestre["periodo_archivo"], por_trimestre["n_registros"], alpha=0.4, label="n registros")
ax1.set_ylabel("n registros")
ax2 = ax1.twinx()
ax2.plot(por_trimestre["periodo_archivo"], por_trimestre["salario_mediano"], color="red", marker="o", label="salario mediano")
ax2.set_ylabel("salario mediano (Q)")
fig.suptitle("Tamaño de muestra y salario mediano por trimestre (2025)")
plt.tight_layout()
plt.show()

por_trimestre


In [ ]:
n_min = por_trimestre["n_registros"].min()
n_max = por_trimestre["n_registros"].max()
sal_ini = por_trimestre.iloc[0]["salario_mediano"]
sal_fin = por_trimestre.iloc[-1]["salario_mediano"]
direccion = "aumenta" if sal_fin > sal_ini else "disminuye" if sal_fin < sal_ini else "se mantiene igual"
print("Interpretación:")
print(f"- La muestra analítica por trimestre varía entre {int(n_min):,} y {int(n_max):,} registros.")
print(f"- Entre el primer y el último trimestre de 2025, el salario mediano {direccion}: Q{sal_ini:,.2f} -> Q{sal_fin:,.2f}.")
print("- La interpretación debe centrarse en cambios descriptivos de los registros analizados, no en estimaciones oficiales de la población.")


## 3. Relaciones entre variables numéricas — (5 pts)

Se calcula la correlación de Pearson entre `salario_mensual`, `edad`, `antiguedad` y
`horas_semanales` usando `VectorAssembler` + `Correlation.corr` de `pyspark.ml.stat`,
sobre **todos los registros elegibles de 2025** (sin muestreo).


In [ ]:
CORR_COLS = ["salario_mensual", "edad", "antiguedad", "horas_semanales"]

df_corr_input = df_2025_analitico.select(*CORR_COLS).na.drop()

va_corr = VectorAssembler(inputCols=CORR_COLS, outputCol="features_corr")
vdf_corr = va_corr.transform(df_corr_input)

corr_matrix = Correlation.corr(vdf_corr, "features_corr", "pearson").head()[0].toArray()
corr_pdf = pd.DataFrame(corr_matrix, index=CORR_COLS, columns=CORR_COLS)
corr_pdf


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr_matrix, vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(CORR_COLS))); ax.set_xticklabels(CORR_COLS, rotation=45, ha="right")
ax.set_yticks(range(len(CORR_COLS))); ax.set_yticklabels(CORR_COLS)
for i in range(len(CORR_COLS)):
    for j in range(len(CORR_COLS)):
        ax.text(j, i, f"{corr_matrix[i, j]:.2f}", ha="center", va="center", color="black")
ax.set_title("Matriz de correlación (Pearson) — 2025")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()


### Respuestas — Ejercicio 3

In [ ]:
corr_salario = corr_pdf.loc["salario_mensual", ["edad", "antiguedad", "horas_semanales"]]
var_mayor = corr_salario.abs().idxmax()
valor_mayor = corr_salario[var_mayor]
corr_edad_ant = corr_pdf.loc["edad", "antiguedad"]
def fuerza_corr(r):
    ar = abs(r)
    if ar < 0.3:
        return "débil"
    if ar < 0.7:
        return "moderada"
    return "fuerte"
print("¿Qué variables presentan mayor asociación lineal con el salario?")
print(f"- La mayor asociación lineal absoluta con salario_mensual es {var_mayor}: r={valor_mayor:.3f} ({fuerza_corr(valor_mayor)}).")
print("- Estas correlaciones describen asociación lineal, no causalidad.")
print("\n¿Existe relación entre edad y antigüedad?")
print(f"- corr(edad, antiguedad) = {corr_edad_ant:.3f} ({fuerza_corr(corr_edad_ant)}).")
print("- Una relación positiva es esperable porque personas de mayor edad han tenido más tiempo para acumular antigüedad, aunque no determina el puesto actual.")


## 4. Segmentación de perfiles mediante KMeans — (10 pts)

**Variables de clustering:** `edad`, `antiguedad`, `horas_semanales`.

**¿Vale la pena incluir salario?** Se prueba también una variante que incluye
`salario_mensual` para decidir, como equipo, si aporta separación adicional a los perfiles
o si domina la solución por su escala/asimetría (aun estandarizando, su alta dispersión y
cola larga puede hacer que los clusters terminen definidos casi exclusivamente por nivel de
ingreso en vez de por el perfil laboral/demográfico). La decisión final y su justificación
se documentan después de comparar ambas variantes.

Todas las variables se estandarizan (media 0, desviación 1) con `StandardScaler` antes de
correr `KMeans`, para que ninguna domine por su escala.


In [ ]:
CLUSTER_COLS_BASE = ["edad", "antiguedad", "horas_semanales"]
CLUSTER_COLS_CON_SALARIO = CLUSTER_COLS_BASE + ["salario_mensual"]
CLUSTER_META_COLS = ["nivel_educativo", "categoria_ocupacional", "dominio", "salario_mensual"]

def preparar_features_cluster(sdf, cols):
    base_cols = list(dict.fromkeys(cols + CLUSTER_META_COLS))
    d = sdf.select(*base_cols).na.drop(subset=cols)
    va = VectorAssembler(inputCols=cols, outputCol="features_raw")
    d = va.transform(d)
    scaler = StandardScaler(inputCol="features_raw", outputCol="features_scaled",
                             withMean=True, withStd=True)
    scaler_model = scaler.fit(d)
    return scaler_model.transform(d)

def evaluar_k(d, variante, cols, ks=(2, 3, 4, 5)):
    resultados = []
    evaluator = ClusteringEvaluator(featuresCol="features_scaled", predictionCol="cluster")
    total = d.count()
    for k in ks:
        km = KMeans(featuresCol="features_scaled", predictionCol="cluster", k=k, seed=SEED)
        modelo = km.fit(d)
        pred = modelo.transform(d).persist()
        tamanos = pred.groupBy("cluster").count().toPandas()
        min_frac = tamanos["count"].min() / total
        resultados.append({
            "variante": variante, "k": k, "cols": cols, "silhouette": evaluator.evaluate(pred),
            "min_frac_cluster": min_frac, "modelo": modelo, "pred": pred
        })
    return resultados

data_A = preparar_features_cluster(df_2025_analitico, CLUSTER_COLS_BASE)
data_B = preparar_features_cluster(df_2025_analitico, CLUSTER_COLS_CON_SALARIO)
resultados_cluster = (
    evaluar_k(data_A, "sin_salario", CLUSTER_COLS_BASE) +
    evaluar_k(data_B, "con_salario", CLUSTER_COLS_CON_SALARIO)
)
comparacion_k = pd.DataFrame([
    {"variante": r["variante"], "k": r["k"], "silhouette": r["silhouette"],
     "min_frac_cluster": r["min_frac_cluster"]}
    for r in resultados_cluster
])
comparacion_k.sort_values(["silhouette", "min_frac_cluster"], ascending=False)


In [ ]:
plt.figure(figsize=(6, 4))
for variante, g in comparacion_k.sort_values("k").groupby("variante"):
    plt.plot(g["k"], g["silhouette"], marker="o", label=variante)
plt.xlabel("k"); plt.ylabel("silhouette score")
plt.title("Silhouette por número de clusters")
plt.legend()
plt.show()


**Criterio de selección del equipo:** se elige el modelo con mayor `silhouette score`
entre las alternativas con clusters de tamaño razonable. Como regla operativa, se exige
que el cluster más pequeño tenga al menos 5% de los registros; si ninguna alternativa
cumple, se toma la mejor puntuación y se documenta la limitación. La comparación incluye
una variante sin salario y otra con salario para evaluar si el ingreso mejora la separación
o domina artificialmente los perfiles.

In [ ]:
MIN_FRAC_CLUSTER = 0.05
candidatos = comparacion_k[comparacion_k["min_frac_cluster"] >= MIN_FRAC_CLUSTER]
if candidatos.empty:
    candidatos = comparacion_k
idx_mejor = candidatos["silhouette"].idxmax()
fila_mejor = comparacion_k.loc[idx_mejor]
mejor_resultado = next(
    r for r in resultados_cluster
    if r["variante"] == fila_mejor["variante"] and r["k"] == int(fila_mejor["k"])
)
MEJOR_VARIANTE = mejor_resultado["variante"]
MEJOR_K = mejor_resultado["k"]
cols_perfil = mejor_resultado["cols"]
pred_final = mejor_resultado["pred"]
print(
    f"Modelo seleccionado: {MEJOR_VARIANTE}, k={MEJOR_K}, "
    f"silhouette={mejor_resultado['silhouette']:.4f}, "
    f"cluster mínimo={mejor_resultado['min_frac_cluster']:.1%}"
)


In [ ]:
# --- Perfil de cada cluster: medias de las variables numéricas + tamaño ---
perfil_num = (pred_final.groupBy("cluster")
              .agg(F.count("*").alias("n"),
                   *[F.mean(c).alias(f"media_{c}") for c in cols_perfil])
              .orderBy("cluster").toPandas())
perfil_num


In [ ]:
# --- Composición categórica y salario por cluster para enriquecer cada perfil ---
pred_con_categoricas = pred_final

composicion_edu = (pred_con_categoricas.groupBy("cluster", "nivel_educativo").count()
                    .orderBy("cluster", "nivel_educativo").toPandas())
composicion_ocup = (pred_con_categoricas.groupBy("cluster", "categoria_ocupacional").count()
                    .orderBy("cluster", "categoria_ocupacional").toPandas())
salario_por_cluster = (pred_con_categoricas.groupBy("cluster")
                        .agg(F.expr("percentile_approx(salario_mensual, 0.5)").alias("salario_mediano"),
                             F.count("*").alias("n"))
                        .orderBy("cluster").toPandas())
display(composicion_edu)
display(composicion_ocup)
display(salario_por_cluster)


### Descripción de los clusters

In [ ]:
perfil_desc = perfil_num.merge(salario_por_cluster[["cluster", "salario_mediano"]], on="cluster", how="left")
for _, row in perfil_desc.iterrows():
    partes = [
        f"edad media={row['media_edad']:.1f}",
        f"antigüedad media={row['media_antiguedad']:.1f} años",
        f"horas semanales medias={row['media_horas_semanales']:.1f}",
        f"salario mediano=Q{row['salario_mediano']:,.2f}",
        f"n={int(row['n']):,}",
    ]
    if "media_salario_mensual" in row.index:
        partes.insert(3, f"salario medio=Q{row['media_salario_mensual']:,.2f}")
    print(f"Cluster {int(row['cluster'])}: " + "; ".join(partes) + ".")
print("\nNota: salario_mensual se incluye como variable de clustering solo si la variante seleccionada fue con_salario; en caso contrario, se usa únicamente para describir los perfiles.")


---
## 5. Pipeline de regresión lineal — (20 pts)

Se estima `salario_mensual` usando exactamente los seis predictores indicados en el enunciado: `edad`, `antiguedad`, `horas_semanales`, `nivel_educativo`, `categoria_ocupacional` y `dominio`.

Para evitar fuga de información, los transformadores (`StringIndexer`, `OneHotEncoder`, `StandardScaler`) y el modelo se ajustan exclusivamente con el conjunto de entrenamiento. Se usa `2025T1`–`2025T3` para entrenamiento y `2025T4` para validación; `2026T1` queda reservado para la evaluación final del Ejercicio 7.


In [ ]:
PREDICTORES_NUM = ["edad", "antiguedad", "horas_semanales"]
PREDICTORES_CAT = ["nivel_educativo", "categoria_ocupacional", "dominio"]
TARGET = "salario_mensual"

train_periodos = ["2025T1", "2025T2", "2025T3"]
valid_periodos = ["2025T4"]

cols_modelo = [TARGET] + PREDICTORES_NUM + PREDICTORES_CAT + ["periodo_archivo"]
df_train_lr = (df_2025_analitico
               .filter(F.col("periodo_archivo").isin(train_periodos))
               .select(*cols_modelo)
               .na.drop(subset=[TARGET] + PREDICTORES_NUM)
               .persist())
df_valid_lr = (df_2025_analitico
               .filter(F.col("periodo_archivo").isin(valid_periodos))
               .select(*cols_modelo)
               .na.drop(subset=[TARGET] + PREDICTORES_NUM)
               .persist())

print(f"Registros entrenamiento LR: {df_train_lr.count():,}")
print(f"Registros validación LR: {df_valid_lr.count():,}")


### Modelo de referencia

El modelo de referencia predice siempre el salario promedio observado en entrenamiento. La regresión lineal debe compararse contra este punto de partida: si no reduce MAE/RMSE o no mejora R², sus predictores no estarían aportando capacidad predictiva útil frente a una regla constante.


In [ ]:
def evaluar_regresion(predicciones, etiqueta):
    evaluadores = {
        "MAE": RegressionEvaluator(labelCol=TARGET, predictionCol="prediction", metricName="mae"),
        "RMSE": RegressionEvaluator(labelCol=TARGET, predictionCol="prediction", metricName="rmse"),
        "R2": RegressionEvaluator(labelCol=TARGET, predictionCol="prediction", metricName="r2"),
    }
    return {"modelo": etiqueta, **{m: ev.evaluate(predicciones) for m, ev in evaluadores.items()}}

salario_medio_train = df_train_lr.agg(F.mean(TARGET).alias("media")).first()["media"]
pred_ref_valid = df_valid_lr.withColumn("prediction", F.lit(float(salario_medio_train)))
metricas_ref = evaluar_regresion(pred_ref_valid, "Referencia: media train")
metricas_ref


### Pipeline y regularización

El pipeline codifica las variables categóricas con `StringIndexer` + `OneHotEncoder`, combina las variables numéricas y dummies con `VectorAssembler`, estandariza los predictores con `StandardScaler` y entrena `LinearRegression`. Como se usa `StandardScaler`, se desactiva la estandarización interna del estimador para no aplicar ambos mecanismos.


In [ ]:
def construir_pipeline_lr(reg_param, elastic_net_param):
    indexers = [
        StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
        for c in PREDICTORES_CAT
    ]
    encoders = [
        OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_oh", handleInvalid="keep")
        for c in PREDICTORES_CAT
    ]
    assembler = VectorAssembler(
        inputCols=PREDICTORES_NUM + [f"{c}_oh" for c in PREDICTORES_CAT],
        outputCol="features_sin_escalar",
        handleInvalid="keep",
    )
    scaler = StandardScaler(
        inputCol="features_sin_escalar",
        outputCol="features",
        withMean=True,
        withStd=True,
    )
    lr = LinearRegression(
        featuresCol="features",
        labelCol=TARGET,
        predictionCol="prediction",
        regParam=reg_param,
        elasticNetParam=elastic_net_param,
        standardization=False,
    )
    return Pipeline(stages=indexers + encoders + [assembler, scaler, lr])

configuraciones_lr = [
    {"nombre": "LR sin regularización", "regParam": 0.0, "elasticNetParam": 0.0},
    {"nombre": "LR Ridge", "regParam": 0.1, "elasticNetParam": 0.0},
    {"nombre": "LR ElasticNet", "regParam": 0.1, "elasticNetParam": 0.5},
]

resultados_lr = []
modelos_lr = {}
for cfg in configuraciones_lr:
    pipeline = construir_pipeline_lr(cfg["regParam"], cfg["elasticNetParam"])
    modelo = pipeline.fit(df_train_lr)
    pred_valid = modelo.transform(df_valid_lr).persist()
    metricas = evaluar_regresion(pred_valid, cfg["nombre"])
    resultados_lr.append({**cfg, **metricas})
    modelos_lr[cfg["nombre"]] = modelo

comparacion_lr = pd.DataFrame([metricas_ref] + resultados_lr)
comparacion_lr = comparacion_lr.sort_values("RMSE")
comparacion_lr


In [ ]:
solo_lr = pd.DataFrame(resultados_lr).sort_values("RMSE")
mejor_cfg_lr = solo_lr.iloc[0].to_dict()
MEJOR_LR_NOMBRE = mejor_cfg_lr["nombre"]
mejor_modelo_lr = modelos_lr[MEJOR_LR_NOMBRE]

print(f"Mejor configuración LR: {MEJOR_LR_NOMBRE}")
print(f"regParam={mejor_cfg_lr['regParam']} | elasticNetParam={mejor_cfg_lr['elasticNetParam']}")
print(f"Validación: MAE={mejor_cfg_lr['MAE']:,.2f} | RMSE={mejor_cfg_lr['RMSE']:,.2f} | R2={mejor_cfg_lr['R2']:.4f}")
print(f"Referencia: MAE={metricas_ref['MAE']:,.2f} | RMSE={metricas_ref['RMSE']:,.2f} | R2={metricas_ref['R2']:.4f}")


In [ ]:
print("Interpretación:")
if mejor_cfg_lr["RMSE"] < metricas_ref["RMSE"]:
    print("- La mejor regresión lineal reduce el RMSE frente al modelo de referencia, por lo que los seis predictores aportan información predictiva en validación.")
else:
    print("- La mejor regresión lineal no mejora el RMSE del modelo de referencia; esto sugiere baja capacidad predictiva lineal con estas variables.")
print(f"- El MAE indica que, en promedio absoluto, las predicciones se alejan Q{mejor_cfg_lr['MAE']:,.2f} del salario observado en 2025T4.")
print(f"- El RMSE (Q{mejor_cfg_lr['RMSE']:,.2f}) penaliza más los errores grandes, relevantes porque el salario suele tener cola derecha.")
print(f"- El R2 de validación ({mejor_cfg_lr['R2']:.4f}) resume la proporción de variabilidad explicada frente a una predicción constante.")
print("- Estos resultados son predictivos y no deben interpretarse como efectos causales de educación, dominio u ocupación sobre salario.")


In [ ]:
ruta_mejor_lr = f"{MODEL_DIR}/mejor_modelo_lr"
mejor_modelo_lr.write().overwrite().save(ruta_mejor_lr)
print("Mejor modelo de regresión lineal guardado en:", ruta_mejor_lr)
